# 1장 — 큰 그림: 시간의 모양을 만드는 네 부품

웹 챕터(1장)와 **같은 모형, 같은 수치**를 파이썬으로 계산합니다.
같은 저량(사용자 수)에 어떤 고리를 끼우느냐에 따라 시간의 모양이 달라지는 것을 확인해요.

> 🤖 **AI 튜터와 함께 공부하세요**: 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch1')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 모형: 신규 가입을 무엇이 정하나

처음 사용자 50명, 36개월. 저량은 **사용자 수 S**, 그리고 지연을 켤 때 쓰는 **보이는 사용자 수 P**(사용자 수를 늦게 따라가는 정보).

| 켠 부품 | 신규 가입(유입) |
|---|---|
| 강화 루프만 | $r \cdot S$ |
| 균형 루프만 | $(K - P)/\tau$ |
| 둘 다 | $r \cdot S \cdot (1 - P/K)$ |
| 지연 | $P$가 $S$를 $D$개월 늦게 따라감: $P$의 변화율 $= (S - P)/D$ |

지연을 끄면 $P = S$(바로 보임). 값: $r = 0.35$/월, $K = 1{,}000$명, $\tau = 4$개월, $D = 3$개월.
계산은 한 달을 16조각(시간 간격 1/16개월)으로 썰어 차례로 쌓습니다. 이 계산법(오일러법)은 5장에서 자세히 배워요.

In [ ]:
S0, r, K, tau, D = 50, 0.35, 1000, 4, 3
T, dt = 36, 1 / 16

def run(R, B, delay):
    # 웹 BehaviorLab(sim.ts behaviorRun)과 같은 계산. 시각 t 와 사용자 수 S 목록을 돌려준다.
    S, P = S0, S0
    ts, Ss = [0.0], [S]
    for i in range(1, int(round(T / dt)) + 1):
        seen = P if delay else S
        if R:
            inflow = r * S * ((1 - seen / K) if B else 1)
        else:
            inflow = (K - seen) / tau if B else 0
        dP = (S - P) / D if delay else 0
        S, P = S + inflow * dt, P + dP * dt
        ts.append(i * dt)
        Ss.append(S)
    return ts, Ss

def at(Ss, month):
    return Ss[int(round(month / dt))]

def peak(ts, Ss):
    i = max(range(len(Ss)), key=lambda k: Ss[k])
    return ts[i], Ss[i]

combos = {
    '변화 없음 (고리 없음)': (False, False, False),
    '지수 성장 (강화만)': (True, False, False),
    '목표 추구 (균형만)': (False, True, False),
    'S자 성장 (강화+균형)': (True, True, False),
    '출렁이며 목표 추구 (균형+지연)': (False, True, True),
    '넘쳤다 출렁이는 S자 (셋 다)': (True, True, True),
}
runs = {name: run(*c) for name, c in combos.items()}
for name, (ts, Ss) in runs.items():
    tp, vp = peak(ts, Ss)
    print(f'{name:22s}  12개월 {at(Ss, 12):>12,.0f}명   36개월 {at(Ss, 36):>14,.0f}명   최고 {vp:,.0f}명({tp:.1f}개월)')

In [ ]:
# 웹 실험실·본문 수치 자가검증
_, Sx = runs['지수 성장 (강화만)']
assert round(at(Sx, 12)) == 3187
assert 12.9e6 < at(Sx, 36) < 13.0e6            # 약 1,294만 명
_, Sg = runs['목표 추구 (균형만)']
assert round(at(Sg, 12)) == 954
_, Ss = runs['S자 성장 (강화+균형)']
assert round(at(Ss, 12)) == 776 and round(at(Ss, 24)) == 996
tp, vp = peak(*runs['출렁이며 목표 추구 (균형+지연)'])
assert round(vp) == 1180 and abs(tp - 9.19) < 0.05
tp, vp = peak(*runs['넘쳤다 출렁이는 S자 (셋 다)'])
assert round(vp) == 1228 and abs(tp - 14.44) < 0.05
_, S0run = runs['변화 없음 (고리 없음)']
assert all(v == 50 for v in S0run)
print('✅ 웹과 일치: 강화만 12개월 3,187명 · 균형+지연 최고 1,180명 · 셋 다 최고 1,228명')

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

colors = ['#94a3b8', '#e11d48', '#2563eb', '#059669', '#d97706', '#7c3aed']
fig, ax = plt.subplots(figsize=(9, 5))
for (name, (ts, Ss)), c in zip(runs.items(), colors):
    ax.plot(ts, Ss, color=c, lw=2.2, label=name)
ax.axhline(K, ls='--', lw=1, color='#2563eb')
ax.set_ylim(0, 1600)
ax.set_xlim(0, 36)
ax.set_xlabel('개월')
ax.set_ylabel('사용자 수(명)')
ax.set_title('같은 저량, 다른 구조, 다른 시간의 모양 (강화만은 그래프 밖으로)')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. 오늘의 발견

- **구조가 행동을 만든다**: 같은 저량이라도 끼운 고리에 따라 시간의 모양이 달라진다.
- 지수 성장 ↔ 강화 루프, 목표 추구 ↔ 균형 루프, S자 ↔ 강화에서 균형으로 지배가 넘어감, 출렁임 ↔ 균형 루프 + 지연.
- 지연은 균형 루프가 보는 정보에 걸려야 효과가 있다(강화 루프만 있을 땐 그대로).
- 그래프 모양은 숨은 구조의 지문. 모양을 보고 고리를 짐작하고, 사람보다 구조를 고친다.

다음 장(2장 욕조의 수학)에서 저량과 유량을 수학으로 정확히 다룹니다.
**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**